In [ ]:
!pip install timm torchattacks pandas Pillow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.25.1 which is incompatible.
bigframes 2.49.0 requires requests>=2.27.1, but you have requests 2.25.1 which is incompatible.
tweepy 4.17.0 requires requests<3,>=2.27.0, but you have requests 2.25.1 which is incompatible.
pysal 25.7 requires requests>=2.27, but you have requests

In [ ]:
!pip install --upgrade requests urllib3 idna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.26.20
    Uninstalling urllib3-1.26.20:
      Successfully uninstalled urllib3-1.26.20
  Attempting uninstall: idna
    Found existing installation: idna 2.10
    Uninstalling idna-2.10:
      Successfully uninstalled idna-2.10
  Attempting uninstall: requests
    Found existing installation: requests 2.25.1
    Uninstalling requests-2.25.1:
      Successfully uninstalled requests-2.25.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchattacks 3.5.1 requires requests~=2.25.1, but you have requests 2.34.2 which is incompatible.
google-colab 1.0.0 requir

чтение картинок, парсит первую строку YOLO-файла, вырезает объект и нормализует его под стандарты ImageNet (224 × 224).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import timm

# 7 классов: 0-Artillery, 1-Missile, 2-Radar, 3-M.Rocket Launcher, 4-Soldier, 5-Tank, 6-Vehicle
NUM_CLASSES = 7
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Код будет работать на: {DEVICE}")

class YoloMilitaryDataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform

        # Фильтруем файлы, чтобы были и картинка, и непустой txt
        all_imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        self.valid_samples = []

        for img in all_imgs:
            lbl = os.path.splitext(img)[0] + '.txt'
            lbl_path = os.path.join(label_dir, lbl)
            if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
                self.valid_samples.append(img)

        print(f"Загружено объектов из {img_dir}: {len(self.valid_samples)}")

    def __len__(self):
        return len(self.valid_samples)

    def __getitem__(self, index):
        img_name = self.valid_samples[index]
        img_path = os.path.join(self.img_dir, img_name)
        lbl_path = os.path.join(self.label_dir, os.path.splitext(img_name)[0] + '.txt')

        image = Image.open(img_path).convert("RGB")
        W, H = image.size

        with open(lbl_path, 'r') as f:
            first_line = f.readline().strip().split()

        class_id = int(first_line[0])
        x_c, y_c, w, h = map(float, first_line[1:])

        # Перевод YOLO координат в пиксели для обрезки
        xmin = max(0, int((x_c - w / 2) * W))
        ymin = max(0, int((y_c - h / 2) * H))
        xmax = min(W, int((x_c + w / 2) * W))
        ymax = min(H, int((y_c + h / 2) * H))

        cropped_image = image.crop((xmin, ymin, xmax, ymax))

        if self.transform:
            cropped_image = self.transform(cropped_image)

        return cropped_image, torch.tensor(class_id)

# Трансформации
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    #transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Инициализация (замени пути, если папки лежат в другом месте)
train_dataset = YoloMilitaryDataset('/content/drive/MyDrive/train/images', '/content/drive/MyDrive/train/labels', transform=train_transform)
val_dataset = YoloMilitaryDataset('/content/drive/MyDrive/val/images', '/content/drive/MyDrive/val/labels', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

Код будет работать на: cuda
Загружено объектов из /content/drive/MyDrive/train/images: 1360
Загружено объектов из /content/drive/MyDrive/val/images: 170


Адаптируем классификаторы под наши 7 выходов. Для примера возьмем ConvNeXt-Nano. Для остальных трех моделей (convnext_base, vit_base_patch16_224, tiny_vit_21m_224) код будет абсолютно идентичным — меняется только строка названия архитектуры.

In [ ]:
def init_model(model_name):
    print(f"Инициализация {model_name}...")
    model = timm.create_model(model_name, pretrained=True)
    model.reset_classifier(num_classes=NUM_CLASSES)
    return model.to(DEVICE)

def train_network(model, model_save_name, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_acc = 0.0

    for epoch in range(epochs):
        model.train()
        train_loss, train_corrects = 0.0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            train_loss += loss.item() * inputs.size(0)
            train_corrects += torch.sum(preds == labels.data)

        # Валидация
        model.eval()
        val_loss, val_corrects = 0.0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data)

        scheduler.step()

        train_acc = train_corrects.double() / len(train_dataset)
        val_acc = val_corrects.double() / len(val_dataset)


        current_lr = optimizer.param_groups[0]['lr']
        print(f"Эпоха {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{model_save_name}.pth")

    print(f"Успешно обучено! Лучшая точность: {best_acc:.4f}\n")

# Запуск обучения первой модели (для теста)
cnext_nano = init_model('convnext_nano')
train_network(cnext_nano, "convnext_nano_military", epochs=15)

Инициализация convnext_nano...
Эпоха 1/15 | Train Acc: 0.2412 | Val Acc: 0.3588
Эпоха 2/15 | Train Acc: 0.5390 | Val Acc: 0.7647
Эпоха 3/15 | Train Acc: 0.7728 | Val Acc: 0.7647
Эпоха 4/15 | Train Acc: 0.8522 | Val Acc: 0.8000
Эпоха 5/15 | Train Acc: 0.9272 | Val Acc: 0.7765
Эпоха 6/15 | Train Acc: 0.9412 | Val Acc: 0.8294
Эпоха 7/15 | Train Acc: 0.9691 | Val Acc: 0.8118
Эпоха 8/15 | Train Acc: 0.9860 | Val Acc: 0.8235
Эпоха 9/15 | Train Acc: 0.9949 | Val Acc: 0.8353
Эпоха 10/15 | Train Acc: 0.9971 | Val Acc: 0.8588
Эпоха 11/15 | Train Acc: 0.9985 | Val Acc: 0.8588
Эпоха 12/15 | Train Acc: 1.0000 | Val Acc: 0.8588
Эпоха 13/15 | Train Acc: 1.0000 | Val Acc: 0.8588
Эпоха 14/15 | Train Acc: 1.0000 | Val Acc: 0.8588
Эпоха 15/15 | Train Acc: 0.9993 | Val Acc: 0.8588
Успешно обучено! Лучшая точность: 0.8588

